# 10-coin similarity matrices from 12 embedding results

This notebook continues the 10-asset pipeline:

1. Load the 12 saved log-return embedding panels.
2. Compute pairwise similarity matrices for each panel using:
   - Wasserstein distance
   - DTW distance
3. Save 24 compressed `npz` outputs plus one manifest table.


In [1]:
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
from numba import njit

EMBEDDING_DIR = Path("/Users/jane/Documents/202511吾-Systems/12.Data0329/10coin_log_return_embeddings")
OUTPUT_ROOT = Path("/Users/jane/Documents/202511吾-Systems/12.Data0329/10coin_similarity_matrices")
WASS_DIR = OUTPUT_ROOT / "wasserstein"
DTW_DIR = OUTPUT_ROOT / "dtw"

WASS_DIR.mkdir(parents=True, exist_ok=True)
DTW_DIR.mkdir(parents=True, exist_ok=True)

embedding_files = sorted(EMBEDDING_DIR.glob("close10_log_return_embedding_m*_tau*.npz"))

print(f"Embedding directory: {EMBEDDING_DIR}")
print(f"Found {len(embedding_files)} embedding files.")
print(f"Output root: {OUTPUT_ROOT}")
embedding_files[:3]


Embedding directory: /Users/jane/Documents/202511吾-Systems/12.Data0329/10coin_log_return_embeddings
Found 12 embedding files.
Output root: /Users/jane/Documents/202511吾-Systems/12.Data0329/10coin_similarity_matrices


[PosixPath('/Users/jane/Documents/202511吾-Systems/12.Data0329/10coin_log_return_embeddings/close10_log_return_embedding_m07_tau1.npz'),
 PosixPath('/Users/jane/Documents/202511吾-Systems/12.Data0329/10coin_log_return_embeddings/close10_log_return_embedding_m07_tau2.npz'),
 PosixPath('/Users/jane/Documents/202511吾-Systems/12.Data0329/10coin_log_return_embeddings/close10_log_return_embedding_m07_tau3.npz')]

In [2]:
@njit
def wasserstein_equal_1d(u, v):
    """Wasserstein-1 distance for two equally weighted 1D empirical samples of equal length."""
    su = np.sort(u)
    sv = np.sort(v)
    total = 0.0
    for k in range(su.shape[0]):
        diff = su[k] - sv[k]
        if diff < 0:
            diff = -diff
        total += diff
    return total / su.shape[0]


@njit
def dtw_abs(u, v):
    """Classical DTW distance with absolute-difference local cost."""
    n = u.shape[0]
    m = v.shape[0]
    dp = np.empty((n + 1, m + 1), dtype=np.float64)
    inf = 1e18

    for i in range(n + 1):
        for j in range(m + 1):
            dp[i, j] = inf

    dp[0, 0] = 0.0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = u[i - 1] - v[j - 1]
            if cost < 0:
                cost = -cost

            best = dp[i - 1, j]
            if dp[i, j - 1] < best:
                best = dp[i, j - 1]
            if dp[i - 1, j - 1] < best:
                best = dp[i - 1, j - 1]

            dp[i, j] = cost + best

    return dp[n, m]


@njit
def compute_similarity_matrices(panel, method_code):
    """Compute all pairwise distance matrices for one embedding panel.

    Parameters
    ----------
    panel : array of shape (n_time, n_coins, m)
    method_code : 0 for Wasserstein, 1 for DTW
    """
    n_time, n_coins, _ = panel.shape
    out = np.zeros((n_time, n_coins, n_coins), dtype=np.float32)

    for t in range(n_time):
        for i in range(n_coins):
            for j in range(i + 1, n_coins):
                if method_code == 0:
                    d = wasserstein_equal_1d(panel[t, i], panel[t, j])
                else:
                    d = dtw_abs(panel[t, i], panel[t, j])

                out[t, i, j] = d
                out[t, j, i] = d

    return out


In [3]:
# Warm up numba compilation once so the main loop is faster.
warm_panel = np.load(embedding_files[0], allow_pickle=True)["embedded_array"][:2].astype(np.float64)
_ = compute_similarity_matrices(warm_panel, 0)
_ = compute_similarity_matrices(warm_panel, 1)
print("Numba functions compiled successfully.")


Numba functions compiled successfully.


In [4]:
pattern = re.compile(r"m(\d+)_tau(\d+)")


def parse_parameters_from_name(path):
    match = pattern.search(path.stem)
    if not match:
        raise ValueError(f"Could not parse window_size and lag from {path.name}")
    return int(match.group(1)), int(match.group(2))


def save_similarity_npz(output_path, similarity_matrices, coins, dates, window_size, lag, method, value_type="log_return"):
    np.savez_compressed(
        output_path,
        similarity_matrices=similarity_matrices.astype(np.float32),
        coins=np.array(coins, dtype=str),
        dates=np.array(dates, dtype=str),
        window_size=np.int64(window_size),
        lag=np.int64(lag),
        method=np.array(method, dtype=str),
        value_type=np.array(value_type, dtype=str),
        n_time=np.int64(similarity_matrices.shape[0]),
        n_coins=np.int64(similarity_matrices.shape[1])
    )


In [5]:
manifest_rows = []

for embedding_path in embedding_files:
    data = np.load(embedding_path, allow_pickle=True)
    panel = data["embedded_array"].astype(np.float64)
    coins = data["coins"].tolist()
    dates = data["dates"].tolist()
    window_size, lag = parse_parameters_from_name(embedding_path)

    print(f"\nProcessing {embedding_path.name} | shape={panel.shape}")

    # Wasserstein
    start = time.time()
    wasserstein_mats = compute_similarity_matrices(panel, 0)
    wasserstein_output = WASS_DIR / f"close10_log_return_wasserstein_similarity_m{window_size:02d}_tau{lag}.npz"
    save_similarity_npz(
        wasserstein_output,
        wasserstein_mats,
        coins=coins,
        dates=dates,
        window_size=window_size,
        lag=lag,
        method="wasserstein"
    )
    wasserstein_time = time.time() - start
    print(f"  Wasserstein saved: {wasserstein_output.name} ({wasserstein_time:.2f}s)")

    manifest_rows.append(
        {
            "embedding_file": embedding_path.name,
            "method": "wasserstein",
            "output_file": wasserstein_output.name,
            "window_size": window_size,
            "lag": lag,
            "n_time": wasserstein_mats.shape[0],
            "n_coins": wasserstein_mats.shape[1],
            "first_date": dates[0],
            "last_date": dates[-1],
            "runtime_seconds": round(wasserstein_time, 4)
        }
    )

    # DTW
    start = time.time()
    dtw_mats = compute_similarity_matrices(panel, 1)
    dtw_output = DTW_DIR / f"close10_log_return_dtw_similarity_m{window_size:02d}_tau{lag}.npz"
    save_similarity_npz(
        dtw_output,
        dtw_mats,
        coins=coins,
        dates=dates,
        window_size=window_size,
        lag=lag,
        method="dtw"
    )
    dtw_time = time.time() - start
    print(f"  DTW saved:         {dtw_output.name} ({dtw_time:.2f}s)")

    manifest_rows.append(
        {
            "embedding_file": embedding_path.name,
            "method": "dtw",
            "output_file": dtw_output.name,
            "window_size": window_size,
            "lag": lag,
            "n_time": dtw_mats.shape[0],
            "n_coins": dtw_mats.shape[1],
            "first_date": dates[0],
            "last_date": dates[-1],
            "runtime_seconds": round(dtw_time, 4)
        }
    )

manifest_df = pd.DataFrame(manifest_rows).sort_values(["method", "window_size", "lag"]).reset_index(drop=True)
manifest_path = OUTPUT_ROOT / "close10_similarity_matrix_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print(f"\nAll similarity matrices saved. Manifest: {manifest_path}")
manifest_df



Processing close10_log_return_embedding_m07_tau1.npz | shape=(1867, 10, 7)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m07_tau1.npz (0.06s)
  DTW saved:         close10_log_return_dtw_similarity_m07_tau1.npz (0.03s)

Processing close10_log_return_embedding_m07_tau2.npz | shape=(1861, 10, 7)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m07_tau2.npz (0.06s)
  DTW saved:         close10_log_return_dtw_similarity_m07_tau2.npz (0.03s)

Processing close10_log_return_embedding_m07_tau3.npz | shape=(1855, 10, 7)


  Wasserstein saved: close10_log_return_wasserstein_similarity_m07_tau3.npz (0.06s)


  DTW saved:         close10_log_return_dtw_similarity_m07_tau3.npz (0.03s)

Processing close10_log_return_embedding_m14_tau1.npz | shape=(1860, 10, 14)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m14_tau1.npz (0.07s)
  DTW saved:         close10_log_return_dtw_similarity_m14_tau1.npz (0.04s)

Processing close10_log_return_embedding_m14_tau2.npz | shape=(1847, 10, 14)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m14_tau2.npz (0.07s)


  DTW saved:         close10_log_return_dtw_similarity_m14_tau2.npz (0.04s)

Processing close10_log_return_embedding_m14_tau3.npz | shape=(1834, 10, 14)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m14_tau3.npz (0.06s)
  DTW saved:         close10_log_return_dtw_similarity_m14_tau3.npz (0.04s)

Processing close10_log_return_embedding_m21_tau1.npz | shape=(1853, 10, 21)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m21_tau1.npz (0.07s)


  DTW saved:         close10_log_return_dtw_similarity_m21_tau1.npz (0.05s)

Processing close10_log_return_embedding_m21_tau2.npz | shape=(1833, 10, 21)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m21_tau2.npz (0.07s)
  DTW saved:         close10_log_return_dtw_similarity_m21_tau2.npz (0.05s)

Processing close10_log_return_embedding_m21_tau3.npz | shape=(1813, 10, 21)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m21_tau3.npz (0.07s)


  DTW saved:         close10_log_return_dtw_similarity_m21_tau3.npz (0.05s)

Processing close10_log_return_embedding_m28_tau1.npz | shape=(1846, 10, 28)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m28_tau1.npz (0.08s)


  DTW saved:         close10_log_return_dtw_similarity_m28_tau1.npz (0.12s)

Processing close10_log_return_embedding_m28_tau2.npz | shape=(1819, 10, 28)
  Wasserstein saved: close10_log_return_wasserstein_similarity_m28_tau2.npz (0.07s)
  DTW saved:         close10_log_return_dtw_similarity_m28_tau2.npz (0.12s)

Processing close10_log_return_embedding_m28_tau3.npz | shape=(1792, 10, 28)


  Wasserstein saved: close10_log_return_wasserstein_similarity_m28_tau3.npz (0.07s)
  DTW saved:         close10_log_return_dtw_similarity_m28_tau3.npz (0.12s)

All similarity matrices saved. Manifest: /Users/jane/Documents/202511吾-Systems/12.Data0329/10coin_similarity_matrices/close10_similarity_matrix_manifest.csv


,embedding_file,method,output_file,window_size,lag,n_time,n_coins,first_date,last_date,runtime_seconds
0,close10_log_return_embedding_m07_tau1.npz,dtw,close10_log_return_dtw_similarity_m07_tau1.npz,7,1,1867,10,2020-08-17,2025-09-26,0.0262
1,close10_log_return_embedding_m07_tau2.npz,dtw,close10_log_return_dtw_similarity_m07_tau2.npz,7,2,1861,10,2020-08-23,2025-09-26,0.0270
2,close10_log_return_embedding_m07_tau3.npz,dtw,close10_log_return_dtw_similarity_m07_tau3.npz,7,3,1855,10,2020-08-29,2025-09-26,0.0265
3,close10_log_return_embedding_m14_tau1.npz,dtw,close10_log_return_dtw_similarity_m14_tau1.npz,14,1,1860,10,2020-08-24,2025-09-26,0.0363
4,close10_log_return_embedding_m14_tau2.npz,dtw,close10_log_return_dtw_similarity_m14_tau2.npz,14,2,1847,10,2020-09-06,2025-09-26,0.0356
5,close10_log_return_embedding_m14_tau3.npz,dtw,close10_log_return_dtw_similarity_m14_tau3.npz,14,3,1834,10,2020-09-19,2025-09-26,0.0354
6,close10_log_return_embedding_m21_tau1.npz,dtw,close10_log_return_dtw_similarity_m21_tau1.npz,21,1,1853,10,2020-08-31,2025-09-26,0.0511
7,close10_log_return_embedding_m21_tau2.npz,dtw,close10_log_return_dtw_similarity_m21_tau2.npz,21,2,1833,10,2020-09-20,2025-09-26,0.0514
8,close10_log_return_embedding_m21_tau3.npz,dtw,close10_log_return_dtw_similarity_m21_tau3.npz,21,3,1813,10,2020-10-10,2025-09-26,0.0503
9,close10_log_return_embedding_m28_tau1.npz,dtw,close10_log_return_dtw_similarity_m28_tau1.npz,28,1,1846,10,2020-09-07,2025-09-26,0.1204


In [6]:
sample_path = WASS_DIR / "close10_log_return_wasserstein_similarity_m21_tau1.npz"
sample = np.load(sample_path, allow_pickle=True)
print("Sample keys:", sample.files)
print("Sample similarity matrix shape:", sample["similarity_matrices"].shape)
print("Coins:", sample["coins"].tolist())
print("First 3 dates:", sample["dates"][:3].tolist())


Sample keys: ['similarity_matrices', 'coins', 'dates', 'window_size', 'lag', 'method', 'value_type', 'n_time', 'n_coins']
Sample similarity matrix shape: (1853, 10, 10)
Coins: ['ETH', 'BTC', 'XRP', 'USDT', 'USDC', 'BNB', 'DOGE', 'ADA', 'TRX', 'SOL']
First 3 dates: ['2020-08-31', '2020-09-01', '2020-09-02']
